## Explore Dataset

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import shapely
import dotenv
import os
import momepy
import networkx as nx

In [2]:
dotenv.load_dotenv()
RIVERS_URL = os.getenv("RIVERS_URL")

## Obtaining data

Keeping all the data in the EPSG 3857 coordinate referencing system (CRS) to work better with distances

### CSO Data

In [3]:
geo_JSON_url = "https://services2.arcgis.com/g6o32ZDQ33GpCIu3/arcgis/rest/services/Thames_Water_Storm_Overflow_Activity_(Production)_view/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
cso: gpd.GeoDataFrame = gpd.read_file(geo_JSON_url)

In [4]:
cso = cso.to_crs(epsg=3857)

In [ ]:
cso.head()

### River data

In [6]:
rivers = gpd.read_file(RIVERS_URL)
rivers = rivers.to_crs(epsg=3857)

In [ ]:
rivers[rivers.Shape_Length == rivers.Shape_Length.max()]

## Start Plotting geometry

### CSO Data

In [ ]:
cso.explore()

### River Data

In [ ]:
m = rivers[rivers["OBJECTID"] == 120644].explore()
cso.explore(m=m)

## Thames analysis

The idea is to try to find CSOs near the river thames and correctly associate them to the rivers, and find their outflow positions

In [10]:
river_thames = rivers[rivers["name1"] == "River Thames"].copy()
river_thames = river_thames.reset_index().drop(columns="index")
all_river_thames = rivers.clip(cso.total_bounds, keep_geom_type=True)
all_river_thames = all_river_thames.explode()

### Method 1 : Using buffer distance ❌ 

In [15]:
def match_points_to_lines(points_gdf: gpd.GeoDataFrame,
                          lines_gdf: gpd.GeoDataFrame,
                          max_distance_meters: float) -> gpd.GeoDataFrame:
    """
    Match points to nearest lines within distance using buffer approach.

    Args:
        points_gdf: GeoDataFrame with Point geometries in EPSG:3857
        lines_gdf: GeoDataFrame with LineString geometries in EPSG:3857
        max_distance_meters: Maximum distance threshold in meters
    """
    # Convert to projected CRS for accurate buffering
    points_projected = points_gdf.to_crs(epsg=3857)
    lines_projected = lines_gdf.to_crs(epsg=3857)

    # Buffer lines by max distance
    buffered_lines = lines_projected.copy()
    buffered_lines.geometry = lines_projected.geometry.buffer(
        max_distance_meters)

    # Spatial join points with buffered lines
    matched_points = gpd.sjoin_nearest(
        points_projected,
        buffered_lines,
        max_distance=max_distance_meters,
        distance_col="distance_meters"
    )

    return matched_points

In [ ]:
bounds_distance = 250  # metres
river_thames_bounds = river_thames.copy()
river_thames_bounds["buffer"] = river_thames.geometry.apply(
    lambda x: x.buffer(bounds_distance))

In [17]:
river_thames_bounds = river_thames_bounds.set_geometry("buffer")

In [ ]:
river_thames_bounds.explore()

In [ ]:
cso_point = 29853
mypoint = cso_thames[cso_thames["OBJECTID"] == cso_point].geometry.to_list()[0]
river_thames_bounds.contains(mypoint).any()

In [21]:
def find_how_many_contains(p: shapely.Point) -> int:
    contains_df = river_thames_bounds.contains(p)
    containing_rivers = river_thames_bounds[contains_df == True]
    return len(containing_rivers)


cso_thames["isClose"] = cso_thames.geometry.apply(
    lambda x: find_how_many_contains(x)
)

In [ ]:
m = cso_thames.explore(column="isClose", cmap='jet')
river_thames.explore(m=m)

In [ ]:
# Important Exception: Point corresponds to 3 rivers
cso_point = 29853
print(cso_thames[cso_thames["OBJECTID"] == cso_point])
m = river_thames_bounds.explore()
cso[cso["OBJECTID"] == cso_point].explore(marker_kwds={"radius": 100}, m=m)

### Method 2: Using gpd.sjoin_nearest ✅

In [11]:
MAX_DISTANCE = 700  # m

In [12]:
nearest_point_info = gpd.sjoin_nearest(
    cso,
    all_river_thames[["geometry", "OBJECTID"]],
    max_distance=MAX_DISTANCE,
    how="left"
)

# Bookeeping
nearest_point_info.drop(columns=["index_right"], inplace=True)
nearest_point_info.rename(columns={"OBJECTID_right": "riverID"}, inplace=True)
nearest_point_info.rename(columns={"OBJECTID_left": "csoID"}, inplace=True)
nearest_point_info = nearest_point_info[~nearest_point_info.index.duplicated(
    keep="first")]

In [13]:
def find_shortest_line(p: shapely.Point, idx: float):
    if not idx > 0:
        return None
    return shapely.shortest_line(p, all_river_thames[all_river_thames["OBJECTID"] == idx].geometry.to_list()[0])


nearest_point_info["connectingLine"] = nearest_point_info.apply(lambda x: find_shortest_line(
    x["geometry"], x["riverID"]), axis=1).set_crs(nearest_point_info.crs)

In [14]:
nearest_point_info["isAttributed"] = nearest_point_info.apply(
    lambda x: x["riverID"] > 0, axis=1)

In [15]:
nearest_point_info["outflowLocation"] = nearest_point_info.connectingLine.apply(
    lambda x: shapely.line_interpolate_point(x, 1, normalized=True)).set_crs(nearest_point_info.crs)

#### Plotting

In [13]:
buffer = all_river_thames.buffer(distance=MAX_DISTANCE)

In [ ]:
m = nearest_point_info.explore(
    column="isAttributed",
    cmap="coolwarm",
    marker_kwds={"radius": 7}
)
# buffer.explore(m=m, highlight=False,popup=False,tooltip=False)
all_river_thames.explore(m=m)
nearest_point_info.connectingLine.explore(
    m=m,
    color="green",
)
nearest_point_info["outflowLocation"].explore(
    m=m,
    color="green"
)

#### Find all rivers in a certain distance of the point

In [16]:
RIVER_DISTANCE = 5_000  # 5km

In [17]:
nearest_point_info_subs = nearest_point_info[(
    nearest_point_info["csoID"] == 31061) ^ (nearest_point_info["csoID"] == 30746)]

In [ ]:
sq_size = 5000  # m
bounds = nearest_point_info_subs.total_bounds
bounds = [
    bounds[0] - sq_size,
    bounds[1] - sq_size,
    bounds[2] + sq_size,
    bounds[3] + sq_size,
]
bounds

In [19]:
riversSub = rivers.clip(bounds).copy().explode()

In [ ]:
riversSub.explore()

In [ ]:
m = nearest_point_info_subs.explore(
    column="isAttributed",
    cmap="coolwarm",
    marker_kwds={"radius": 7}
)
nearest_point_info["connectingLine"].explore(m=m, color="green")
nearest_point_info_subs.outflowLocation.explore(
    m=m,
    color="orange",
    marker_kwds={"radius": 7},
)
riversSub.explore(m=m)

## Risk Assignment

In [104]:
DISTANCE_CUTOFF = 2000

In [274]:
csos = nearest_point_info[["geometry", "csoID", "outflowLocation"]]
rivers_df = all_river_thames

### Converting the rivers to a graph

In [282]:
# Dividing each river segment into equally spaced points

DIVISION_DISTANCE = 125  # metres.


def split_linestring(l: shapely.LineString) -> list[shapely.LineString]:

    points: np.ndarray[shapely.Point] = shapely.line_interpolate_point(l,
                                                                       np.arange(
                                                                           0, l.length, DIVISION_DISTANCE),
                                                                       # np.linspace(0,1,n_divisions),
                                                                       # normalized=True,
                                                                       )

    endpoint = shapely.Point(l.coords[-1])
    if endpoint not in points:
        points = np.append(points, endpoint)

    return [shapely.LineString([p1, p2]) for p1, p2 in zip(points[:-1], points[1:])]


riverThamesExpl = rivers_df.copy()
riverThamesExpl["geometry2"] = riverThamesExpl.geometry.apply(split_linestring)

riverThamesExpl = riverThamesExpl[[
    "OBJECTID", "geometry2"]].explode("geometry2")
riverThamesExpl.rename(
    columns={"OBJECTID": "riverID", "geometry2": "geometry"}, inplace=True)

riverThamesExpl.geometry = riverThamesExpl.geometry.astype("geometry")
riverThamesExpl = gpd.GeoDataFrame(riverThamesExpl)
riverThamesExpl.geometry = riverThamesExpl.geometry.set_crs(rivers_df.crs)
riverSplit = riverThamesExpl.reset_index()

NOTE: Assuming the order of linestring is the direction of flow of the river

In [283]:
riverGraph = momepy.gdf_to_nx(
    riverSplit,
    approach="primal",
    directed=True,  # To indicate flow direction
    multigraph=False,
)

In [284]:
from collections import defaultdict

# Create a dictionary to store edge IDs for each node
node_edge_ids = defaultdict(set)

# Populate the dictionary by iterating over edges
for u, v, data in riverGraph.edges(data=True):
    # Assuming 'edgeID' is stored in edge attributes
    edge_id = data.get('riverID')
    node_edge_ids[u].add(edge_id)
    node_edge_ids[v].add(edge_id)
nx.set_node_attributes(riverGraph, node_edge_ids, name="riverID")

In [ ]:
positions = {n: [n[0], n[1]] for n in list(riverGraph.nodes)}

# Plot
f, ax = plt.subplots(figsize=[10, 10])
nx.draw_networkx_edges(riverGraph, positions, ax=ax,
                       node_size=5, arrowstyle='->')
ax.scatter(
    nearest_point_info_subs["outflowLocation"].apply(
        lambda x: x.coords[0][0]).to_list(),
    nearest_point_info_subs["outflowLocation"].apply(
        lambda x: x.coords[0][1]).to_list(),
)
plt.show()

### Add CSOs to the river graph

In [285]:
CSO_DISTANCE_MAX = 700


def return_coords(x, df):
    if pd.isna(x):
        return None
    else:
        return tuple(df.loc[x, "geometry"].coords)


def add_closest_edge(csoDf: gpd.GeoDataFrame, riversDf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:

    # Find nearest river
    nearest = gpd.sjoin_nearest(
        csoDf, riversDf[["geometry", "riverID"]], how="left", max_distance=CSO_DISTANCE_MAX)

    # Remove duplicates
    nearest = nearest[~nearest.index.duplicated(keep="first")]

    # Add the edge info
    nearest["closestEdge"] = nearest["index_right"].apply(
        lambda x: return_coords(x, riversDf))

    # Remove unused columns
    nearest.drop(columns=["index_right", "riverID"], inplace=True)

    return nearest

In [286]:
csos = add_closest_edge(csos, riverSplit)

In [ ]:
csos

TO CLEAN 👇

In [288]:
def add_cso_to_graph(graph: nx.DiGraph, p: shapely.Point, ID: int, edge: tuple) -> None:
    csoInfoName = "csoInfo"
    # Loop over successors and predecessors, add data

    next_node = edge[1]
    # Find distance between CSO and first point
    next_dist = p.distance(shapely.Point(next_node))
    curr_nodes = []

    # Check if distance lesser than threshold
    if next_dist < DISTANCE_CUTOFF:
        curr_nodes.append(next_node)
        # Get prev data
        dataBefore = graph.nodes(data=csoInfoName)[next_node]
        # If exists, add data
        if dataBefore is not None and isinstance(dataBefore, dict):
            dataBefore[ID] = next_dist
        # If doesn't exist, add new dict
        else:
            nx.set_node_attributes(
                graph, {next_node: {ID: next_dist}}, csoInfoName)

    # Loop over children nodes

    while len(curr_nodes) > 0:
        new_curr = []
        for node in curr_nodes:
            # Check children

            for child in graph.successors(node):

                # Check if already added
                dataBefore = graph.nodes(data=csoInfoName)[child]

                if dataBefore is not None and ID in dataBefore:
                    continue

                # Check distance
                total_dist = graph.nodes(data=csoInfoName)[
                    node][ID] + graph[node][child]["mm_len"]
                if total_dist >= DISTANCE_CUTOFF:
                    continue

                # Add data

                if dataBefore is not None and isinstance(dataBefore, dict):
                    dataBefore[ID] = total_dist
                    nx.set_node_attributes(
                        graph, {child: dataBefore}, csoInfoName)

                # If doesn't exist, add new dict
                else:
                    nx.set_node_attributes(
                        graph, {child: {ID: total_dist}}, csoInfoName)

                # Add to new_curr
                new_curr.append(child)

        curr_nodes = new_curr
    return

In [289]:
for row in csos.iterrows():
    outflowLoc = row[1]["outflowLocation"]
    outflowID = row[1]["csoID"]
    edge = row[1]["closestEdge"]
    if edge is not None:
        # Add CSO to graph
        add_cso_to_graph(riverGraph, outflowLoc, outflowID, edge)

In [290]:
points_data = momepy.nx_to_gdf(riverGraph, points=True, lines=False)
points_data.drop(columns=["x", "y", "nodeID"], inplace=True)

In [291]:
def convert_to_list(x: dict):
    if isinstance(x, dict):
        return list(zip(x.keys(), x.values()))
    elif isinstance(x, set):
        return list(x)
    elif isinstance(x, list):
        return x
    elif pd.isna(x):
        return []
    else:
        return x


points_data["csoInfo"] = points_data["csoInfo"].apply(convert_to_list)
points_data.riverID = points_data.riverID.apply(convert_to_list)

In [ ]:
points_data

### Assign Risk according to CSOs

In [293]:
def inv_dist(csos: list[tuple]) -> float:
    max_val = 0
    for _, dist in csos:
        max_val = max(max_val, 2000-dist)
    return max_val

In [ ]:
points_data.crs

In [294]:
points_data["Risk"] = points_data.csoInfo.apply(inv_dist)

In [ ]:
m = points_data.explore(column="Risk",cmap = "gnuplot")
csos.explore(m=m, marker_kwds={"marker_size"=7})